> Disclaimer
>
> This notebook is not intended to be used and was created in the process of developping Bob.
> Some or many features may have been modified since the creation of this file.
> Use at your own risk. 
>
> Christian Tremblay

In [1]:
from pathlib import Path

from typing import Any, Dict

from matplotlib.backend_bases import MouseEvent
from sympy import vfield

from bob.core import (
    Equipment,
    System,
    get_datagraph,
    bind_model_namespace,
    dump,
)
from bob.equipment.hvac.compressor import AirCompressor
from bob.equipment.hvac.damper import PneumaticDamperActuator
from bob.equipment.hvac.coil import ChilledWaterCoil, HotWaterCoil, WaterCoil
from bob.equipment.hvac.fan import Fan
from bob.equipment.hvac.filter import Filter
from bob.equipment.hvac.damper import Window, Damper, DamperActuator
from bob.equipment.hvac.heatexchanger import Accumulator, Accumulator4SidesDuct
from bob.equipment.hvac.humidifier import SteamPipe, Humidifier
from bob.equipment.lighting.light import Light
from bob.equipment.electricity.vfd import VFD
from bob.equipment import contains_Equipment_list

from bob.equipment.hvac.airhandlingunit import AirHandlingUnit
from bob.equipment.hvac.vav import VAV

from bob.sensor.temperature import AirTemperatureSensor, WaterTemperatureSensor
from bob.sensor.humidity import AirHumiditySensor
from bob.sensor.pressure import DifferentialStaticPressureSensor
from bob.sensor.flow import AirFlowSensor
from bob.sensor.gas import CO2Sensor
from bob.sensor import define_sensors


from bob.space.physical import Building, Floor, MechanicalRoom, Roof, Office, Room, Bathroom, Corridor
from bob.space.hvac import HVACSpace, HVACZone
from bob.space.light import LightingSpace, LightingZone

from bob.connections.air import *

from bob.datasource.external import BACnetReference, NiagaraORDReference

SAMPLE_HEADER = """# baseURI: http://data.ashrae.org/standard223/1.0/sample/{sample_name}
# imports: http://data.ashrae.org/standard223/1.0/model/all

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://data.ashrae.org/standard223/1.0/data/{sample_name}>
  a owl:Ontology ;
  rdfs:isDefinedBy <http://data.ashrae.org/standard223/1.0/sample/{sample_name}> ;
  rdfs:label "{sample_name}" ;
  owl:imports <http://data.ashrae.org/standard223/1.0/model/all> .

"""


def sample_header(sample_name):
    """Prints the sample header."""
    # sys.stdout.write(SAMPLE_HEADER.format(sample_name=sample_name))
    return SAMPLE_HEADER.format(sample_name=sample_name)

model_name = Path('ZooPavillonDesElephants').stem
__namespace__ = bind_model_namespace("zoo", f"urn:zoo/{model_name}/")

# Physical spaces
building = Building(label='B-1', comment="Pavillon des éléphants")
floor1 = Floor(label='RdC', comment="Rez-de-chaussée")
mezzanine = Floor(label="mezz", comment="Mezzanine")
mechroom = MechanicalRoom(label='MechRoom', comment="Salle mécanique")
mechroom_chiller = MechanicalRoom(label='MechRoom', comment="Salle mécanique des refroidisseurs")
office = Office(label='Office', comment='Bureau des gardiens')
entrepot = Room(label='Entrepot', comment='Entrepot à foin')
enclos_elephants = Room(label='Box1', comment="Enclos")
enclos_girafes = Room(label='Box2', comment='Enclos des girafes')

# HVAC Spaces
enclos_elephants_hvac = HVACSpace(label='enclos_elephants_hvac')
enclos_girafe_hvac = HVACSpace(label='enclos_girafe_hvac')

# Outdoor
outdoor = AirConnection(label='Outdoor')

# Worth defining a system to represent the Heat Exhanger
# This process needs to be improved... should be a simple class like custom system
# taking a template for configuration
hx = System(label='ECHANGEUR', comment="The complete heat Exchanger by Trane")
hx_outdoorCP1=AirBidirectionalSystemConnectionPoint(hx),
hx_outdoorCP2=AirBidirectionalSystemConnectionPoint(hx),
hx_returnDuct=AirInletSystemConnectionPoint(hx),
hx_supplyDuct=AirOutletSystemConnectionPoint(hx),
hx_pneumaticInlet=CompressedAirInletSystemConnectionPoint(hx)



# System - Air Handling Unit
accumulator1 = Accumulator(label='Acc1', comment='Accumulator #1')
accumulator2 = Accumulator(label='Acc2', comment='Accumulator #2')
acc1_damper_a = Damper(label='ACC1_DPR-A', comment="Accumulator #1 Damper A")
acc1_damper_b = Damper(label='ACC1_DPR-B', comment="Accumulator #1 Damper B")
acc1_damper_c = Damper(label='ACC1_DPR-C', comment="Accumulator #1 Damper C")
acc1_damper_d = Damper(label='ACC1_DPR-D', comment="Accumulator #1 Damper D")
acc2_damper_a = Damper(label='ACC2_DPR-A', comment="Accumulator #2 Damper A")
acc2_damper_b = Damper(label='ACC2_DPR-B', comment="Accumulator #2 Damper B")
acc2_damper_c = Damper(label='ACC2_DPR-C', comment="Accumulator #2 Damper C")
acc2_damper_d = Damper(label='ACC2_DPR-D', comment="Accumulator #2 Damper D")

acc_4sides_duct = Accumulator4SidesDuct(label='Accumulator 4 sides duct', comment="It contains a Air Connection to connect 4 sides and a pneumatic damper")
acc_4sides_damper = PneumaticDamperActuator(label='Accumulator 4 sides damper', comment="This damper switch the side of the airflow going in accumulator 1 & 2")

hx > [accumulator1, accumulator2, acc1_damper_a, acc1_damper_b, acc1_damper_c, acc1_damper_d, acc2_damper_a, acc2_damper_b, acc2_damper_c, acc2_damper_d, acc_4sides_damper, acc_4sides_duct]

filters = Filter(label='FLT')

coil = WaterCoil(label='SE-1', comment='This coil acts as a cooling coil in summer, heating coil in winter')
sf = Fan(label='UV-1', comment='Supply Fan')
supply_duct = AirConnection(label='SupplyDuct', comment='This is where 3 duct are connected going to Girafes, Elephants and UV-3 (Manège)')
return_duct = AirConnection(label='ReturnDuct', comment='This is where 2 ducts are connected coming from Girafes and Elephants')
av2 = Damper(label='AV-2', comment="Damper going to Girafe")
av4 = Damper(label='AV-4', comment="Damper going to Éléphants")
av1 = Damper(label='AV-1', comment="Damper Coming from Girafe (return)")
av3 = Damper(label='AV-3', comment="Damper coming from Éléphants (return)")
rf = Fan(label='VR-1', comment="Return Fan")

vfd_sf = VFD(label='VFD-1', comment='VFD for Supply Fan')
vfd_rf = VFD(label='VFD-2', comment='VFD for return fan')

hum = Humidifier(label='HUM-1', comment='Humidifier')
hum_pipe = SteamPipe(label='HUM-1_Buse', comment='The pipe connected in the duct to provide humidity in air')
#hum.steamOutlet >> hum_pipe.steamInlet

aircomp = AirCompressor(label='ACOMP-1', comment="Air Compressor")
aircomp.compressedAirOutlet >> acc_4sides_damper.compressedAirInlet


# Sensors
te1 = AirTemperatureSensor(label='TE-1', comment='Outdoor air preheated by exhanger', hasExternalReference=BACnetReference('bacnet://345/analog-value/1/present-value'))
ha1 = AirHumiditySensor(label='HA-1')
tpd1 = DifferentialStaticPressureSensor(label='TPD-1', comment="Filters differential pressure")
taec1 = WaterTemperatureSensor(label='TAEC-1', comment='Water temperature feeding coil')
tbl1 = Equipment(label='TBL-1', comment='Freeze Thermostat')
ta1 = AirTemperatureSensor(label='TA-1', comment='Discharge Air Temperature Sensor')
fs1 = Equipment(label="FS-1", comment="Air flow switch for humidifier")
hlh1 = Equipment(label='HLH-1', comment="Humidity High Level Stat")
tpd2 = DifferentialStaticPressureSensor(label='TPD-2', comment="Static Discharge Air Pressure Sensor")
co2_1 = CO2Sensor(label='CO2-1', comment='Return Air CO2 Sensor (Elephants)')
co2_2 = CO2Sensor(label='CO2-2', comment='Return Air CO2 Sensor (Girafes)')
hr1 = AirHumiditySensor(label='HR-1', comment="Return Air Humidity Sensor")
tr1 = AirTemperatureSensor(label='TR-1', comment="Return Air Temperature Sensor")

# Connections
outdoor >> accumulator1.outdoorSide
outdoor >> accumulator2.outdoorSide
after_accumulator1 = AirConnection(label='AfterAcc1', comment='After Accumulator 1, there are 4 dampers to control air flow, need a connection to connect those 4 dampers')
after_accumulator2 = AirConnection(label='AfterAcc2', comment='After Accumulator 2, there are 4 dampers to control air flow, need a connection to connect those 4 dampers')

after_dampers1 = AirConnection(label='AfterDpr1', comment='4 dampers are feeding the 4 sided duct of the exchanger')
after_dampers2 = AirConnection(label='AfterDpr2', comment='4 dampers are feeding the 4 sided duct of the exchanger')

accumulator1.indoorSide >> after_accumulator1 >> acc1_damper_a >> after_dampers1
after_accumulator1 >> acc1_damper_b >> after_dampers1
after_accumulator1 >> acc1_damper_c >> after_dampers1
after_accumulator1 >> acc1_damper_d >> after_dampers1
accumulator2.indoorSide >> after_accumulator2 >> acc2_damper_a >> after_dampers2
after_accumulator2 >> acc2_damper_b >> after_dampers2
after_accumulator2 >> acc2_damper_c >> after_dampers2
after_accumulator2 >> acc2_damper_d >> after_dampers2

after_dampers1 >> acc_4sides_duct.accumulator1Connection
after_dampers2 >> acc_4sides_duct.accumulator2Connection

acc_4sides_duct.supplyDuctOutlet >> filters.airInlet
filters.airOutlet >> coil.airInlet
coil.airOutlet >> sf.airInlet
sf.airOutlet >> hum_pipe.airInlet
hum_pipe.airOutlet >> supply_duct
supply_duct >> av2.airInlet
supply_duct >> av4.airInlet

av2.airOutlet >> enclos_girafe_hvac.ductAirInlet
av4.airOutlet >> enclos_elephants_hvac.ductAirInlet

enclos_girafe_hvac.ductAirOutlet >> av1.airInlet 
av1.airOutlet >> return_duct
enclos_elephants_hvac.ductAirOutlet >> av3.airInlet
av3.airOutlet >> return_duct
return_duct >> rf.airInlet
rf.airOutlet >> acc_4sides_duct.returnDuctInlet

# Localise sensors
te1 % acc_4sides_duct.supplyDuctOutlet
te1.hasPhysicalLocation = mechroom
ha1 % acc_4sides_duct.supplyDuctOutlet
ha1.hasPhysicalLocation = mechroom
ta1 % supply_duct
ta1.hasPhysicalLocation = mechroom
tpd1['highPort'] % filters.airInlet
tpd1['lowPort'] % filters.airOutlet
tpd1.hasPhysicalLocation = mechroom
tpd2['highPort'] % supply_duct
tpd2['lowPort'] % enclos_elephants_hvac
tpd2.hasPhysicalLocation = mechroom
co2_1 % enclos_girafe_hvac.ductAirOutlet
co2_1.hasPhysicalLocation = enclos_elephants
co2_2 % enclos_elephants_hvac.ductAirOutlet
co2_2.hasPhysicalLocation = enclos_elephants
hr1 % return_duct
hr1.hasPhysicalLocation = mechroom
tr1 % return_duct
tr1.hasPhysicalLocation = mechroom



In [2]:
te1.observedProperty

{'node': rdflib.term.URIRef('urn:zoo/ZooPavillonDesElephants/00140'), 'label': 'TE-1.Measure', 'comment': '', 'hasValue': None, 'hasExternalReference': {'node': rdflib.term.URIRef('urn:zoo/ZooPavillonDesElephants/00138'), 'label': '', 'comment': '', 'hasRef': rdflib.term.Literal('bacnet://345/analog-value/1/present-value')}, 'hasQuantityKind': rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Temperature'), 'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/DEG_C'), 'ofSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223#Medium-Air'), 'label': '', 'comment': ''}}